# Advanced Problems with Solutions: Making Reusable Iterables from Generators

This notebook focuses on the practical problem behind generator functions: a generator object is an **iterator**, so it is consumed after one pass. To make something reusable, build an **iterable** whose `__iter__` method returns a *fresh iterator* every time.

Core rule:

```python
iterable.__iter__()  -> returns a new iterator each time
iterator.__iter__()  -> usually returns itself
```

Each problem below includes a prompt, solution, checks, and extra examples.

In [1]:
from collections.abc import Iterable, Iterator
from itertools import islice, count, tee
from dataclasses import dataclass
from typing import Callable, Any, Optional


def section(title: str) -> None:
    print("\n" + "=" * len(title))
    print(title)
    print("=" * len(title))

## Problem 1 — Diagnose Exhaustion and Build a Reusable Squares Iterable

**Task**

1. Write a generator function `squares_gen(n)` that yields square numbers from `0` through `(n-1) ** 2`.
2. Show that a generator object is exhausted after one full pass.
3. Create a class `Squares` that is reusable: `list(Squares(5))` should work repeatedly.
4. Add basic validation and a helpful `repr`.

**Best practice**: store the *generator recipe* or parameters, not a generator object that will be reused accidentally.

In [2]:
def squares_gen(n: int):
    if not isinstance(n, int):
        raise TypeError("n must be an int")
    if n < 0:
        raise ValueError("n must be >= 0")
    for i in range(n):
        yield i ** 2


section("Generator exhaustion")
g = squares_gen(5)
print(list(g))
print(list(g))  # empty: same generator object is exhausted


class Squares:
    def __init__(self, n: int):
        if not isinstance(n, int):
            raise TypeError("n must be an int")
        if n < 0:
            raise ValueError("n must be >= 0")
        self.n = n

    def __iter__(self):
        # New generator object every time iter(Squares(...)) is called.
        return squares_gen(self.n)

    def __len__(self):
        return self.n

    def __repr__(self):
        return f"Squares(n={self.n})"


section("Reusable iterable")
sq = Squares(5)
print(sq)
print(list(sq))
print(list(sq))

assert isinstance(sq, Iterable)
assert not isinstance(sq, Iterator)
assert list(sq) == [0, 1, 4, 9, 16]
assert list(sq) == [0, 1, 4, 9, 16]
assert len(sq) == 5


Generator exhaustion
[0, 1, 4, 9, 16]
[]

Reusable iterable
Squares(n=5)
[0, 1, 4, 9, 16]
[0, 1, 4, 9, 16]


## Problem 2 — Write a Generic `IterableFromGenerator` Adapter

**Task**

Create a reusable adapter class that accepts a generator function plus arguments. Each call to `__iter__` should call the generator function again.

Example target:

```python
numbers = IterableFromGenerator(range_like, 2, 10, step=2)
list(numbers)  # [2, 4, 6, 8]
list(numbers)  # [2, 4, 6, 8]
```

**Best practice**: validate that `factory(*args, **kwargs)` returns an iterator.

In [3]:
class IterableFromGenerator:
    def __init__(self, factory: Callable[..., Iterator], *args: Any, **kwargs: Any):
        if not callable(factory):
            raise TypeError("factory must be callable")
        self.factory = factory
        self.args = args
        self.kwargs = dict(kwargs)

    def __iter__(self):
        iterator = self.factory(*self.args, **self.kwargs)
        if not isinstance(iterator, Iterator):
            raise TypeError("factory must return an iterator")
        return iterator

    def __repr__(self):
        name = getattr(self.factory, "__name__", self.factory.__class__.__name__)
        parts = [repr(arg) for arg in self.args]
        parts += [f"{key}={value!r}" for key, value in self.kwargs.items()]
        return f"IterableFromGenerator({name}, {', '.join(parts)})"


def range_like(start: int, stop: int, step: int = 1):
    current = start
    if step == 0:
        raise ValueError("step cannot be zero")
    if step > 0:
        while current < stop:
            yield current
            current += step
    else:
        while current > stop:
            yield current
            current += step


section("Generic generator-to-iterable adapter")
evens = IterableFromGenerator(range_like, 2, 10, step=2)
print(evens)
print(list(evens))
print(list(evens))

assert list(evens) == [2, 4, 6, 8]
assert list(IterableFromGenerator(range_like, 5, 0, step=-2)) == [5, 3, 1]


Generic generator-to-iterable adapter
IterableFromGenerator(range_like, 2, 10, step=2)
[2, 4, 6, 8]
[2, 4, 6, 8]


## Problem 3 — Shared State Bug with `enumerate`, `zip`, and Generator Objects

**Task**

A developer creates `g = squares_gen(6)` and then creates `enum_g = enumerate(g)`. They consume from `g` first, then from `enum_g`.

1. Explain the bug using code.
2. Fix it by using the reusable `Squares` iterable.
3. Demonstrate that `enumerate(Squares(6))` receives a fresh iterator.

**Best practice**: pass iterables into pipelines when you need repeatability; pass iterators when one-shot streaming is intentional.

In [4]:
section("Shared state bug")
g = squares_gen(6)
enum_g = enumerate(g)

print(next(g))       # 0
print(next(g))       # 1
print(next(enum_g))  # index is 0, but value is 4: confusing shared state


section("Fixed with reusable iterable")
squares = Squares(6)
enum_a = enumerate(squares)
enum_b = enumerate(squares)

print(next(enum_a))
print(next(enum_a))
print(next(enum_b))  # starts from the beginning, independent of enum_a

assert next(enumerate(Squares(3))) == (0, 0)
assert list(enumerate(Squares(3))) == [(0, 0), (1, 1), (2, 4)]


Shared state bug
0
1
(0, 4)

Fixed with reusable iterable
(0, 0)
(1, 1)
(0, 0)


## Problem 4 — Implement Independent Concurrent Iteration

**Task**

Create an iterable `Countdown(start)` that counts from `start` down to `0`. Then prove that two active iterators over the same iterable do **not** share state.

Expected behavior:

```python
c = Countdown(3)
a = iter(c)
b = iter(c)
next(a)  # 3
next(a)  # 2
next(b)  # 3, not 1
```

**Best practice**: `__iter__` should not store iteration position on `self` for reusable iterables.

In [5]:
class Countdown:
    def __init__(self, start: int):
        if start < 0:
            raise ValueError("start must be >= 0")
        self.start = start

    def __iter__(self):
        current = self.start
        while current >= 0:
            yield current
            current -= 1


section("Independent concurrent iteration")
c = Countdown(3)
a = iter(c)
b = iter(c)

print(next(a))
print(next(a))
print(next(b))
print(list(a))
print(list(b))

assert next(iter(c)) == 3
assert list(c) == [3, 2, 1, 0]
assert list(c) == [3, 2, 1, 0]


Independent concurrent iteration
3
2
3
[1, 0]
[2, 1, 0]


## Problem 5 — Anti-Pattern: An Iterable That Accidentally Behaves Like an Iterator

**Task**

The class below stores iteration state on `self`. Identify why it is not reusable, then fix it.

```python
class BadCountdown:
    def __iter__(self):
        return self
    def __next__(self):
        ...
```

**Best practice**: separate iterable objects from iterator objects unless you intentionally want one-shot behavior.

In [6]:
class BadCountdown:
    def __init__(self, start: int):
        self.current = start

    def __iter__(self):
        return self

    def __next__(self):
        if self.current < 0:
            raise StopIteration
        value = self.current
        self.current -= 1
        return value


section("Bad one-shot countdown")
bad = BadCountdown(3)
print(list(bad))
print(list(bad))  # empty because bad is its own exhausted iterator


class GoodCountdown:
    def __init__(self, start: int):
        self.start = start

    def __iter__(self):
        current = self.start
        while current >= 0:
            yield current
            current -= 1


section("Good reusable countdown")
good = GoodCountdown(3)
print(list(good))
print(list(good))

assert list(good) == [3, 2, 1, 0]
assert iter(good) is not good
assert iter(BadCountdown(1)) is not iter(good)


Bad one-shot countdown
[3, 2, 1, 0]
[]

Good reusable countdown
[3, 2, 1, 0]
[3, 2, 1, 0]


## Problem 6 — Build a Reusable Lazy Data Pipeline

**Task**

Given raw text lines, create an iterable `CleanIntegers` that:

1. Strips each line.
2. Ignores blank lines and comments starting with `#`.
3. Converts valid lines to integers.
4. Raises a helpful error for invalid data.
5. Is reusable.

**Best practice**: pipeline stages can be generator functions; the class stores the source data or source factory and returns a new pipeline each time.

In [7]:
def clean_integer_lines(lines):
    for line_number, raw_line in enumerate(lines, start=1):
        line = raw_line.strip()
        if not line or line.startswith("#"):
            continue
        try:
            yield int(line)
        except ValueError as exc:
            raise ValueError(f"line {line_number}: expected integer, got {raw_line!r}") from exc


class CleanIntegers:
    def __init__(self, lines):
        # Store a tuple snapshot so repeated iteration is stable.
        self._lines = tuple(lines)

    def __iter__(self):
        return clean_integer_lines(self._lines)


section("Reusable lazy cleaning pipeline")
raw = [" 10 ", "", "# ignore me", "20", "-5"]
clean = CleanIntegers(raw)
print(list(clean))
print(sum(clean))
print(max(clean))

assert list(clean) == [10, 20, -5]
assert sum(clean) == 25

try:
    list(CleanIntegers(["1", "oops", "3"]))
except ValueError as err:
    print("Caught:", err)


Reusable lazy cleaning pipeline
[10, 20, -5]
25
20
Caught: line 2: expected integer, got 'oops'


## Problem 7 — Snapshot vs Live Iterables

**Task**

Implement two iterable classes over a mutable list:

1. `LiveIterable(data)` should see changes made to `data` before iteration.
2. `SnapshotIterable(data)` should freeze the data at construction time.

Then demonstrate the difference.

**Best practice**: document whether your iterable has live or snapshot semantics.

In [8]:
class LiveIterable:
    def __init__(self, data):
        self.data = data

    def __iter__(self):
        for item in self.data:
            yield item


class SnapshotIterable:
    def __init__(self, data):
        self.data = tuple(data)

    def __iter__(self):
        for item in self.data:
            yield item


section("Live vs snapshot semantics")
values = [1, 2, 3]
live = LiveIterable(values)
snapshot = SnapshotIterable(values)

values.append(4)

print("live:", list(live))
print("snapshot:", list(snapshot))

assert list(live) == [1, 2, 3, 4]
assert list(snapshot) == [1, 2, 3]


Live vs snapshot semantics
live: [1, 2, 3, 4]
snapshot: [1, 2, 3]


## Problem 8 — Lazy Batching Iterable with `yield from`

**Task**

Create `Batches(iterable, size)` that returns batches as tuples. It must be reusable when the input is reusable.

Example:

```python
list(Batches(range(7), 3))
# [(0, 1, 2), (3, 4, 5), (6,)]
```

Then create `Flattened(iterable_of_iterables)` using `yield from`.

**Best practice**: use `iter(source)` inside `__iter__`, not in `__init__`, to avoid prematurely creating and storing a one-shot iterator.

In [9]:
class Batches:
    def __init__(self, iterable, size: int):
        if size <= 0:
            raise ValueError("size must be positive")
        self.iterable = iterable
        self.size = size

    def __iter__(self):
        iterator = iter(self.iterable)
        while True:
            batch = tuple(islice(iterator, self.size))
            if not batch:
                return
            yield batch


class Flattened:
    def __init__(self, iterable_of_iterables):
        self.iterable_of_iterables = iterable_of_iterables

    def __iter__(self):
        for inner in self.iterable_of_iterables:
            yield from inner


section("Batches and flattening")
batches = Batches(range(7), 3)
print(list(batches))
print(list(batches))

flat = Flattened(batches)
print(list(flat))

assert list(Batches(range(7), 3)) == [(0, 1, 2), (3, 4, 5), (6,)]
assert list(Flattened(Batches(range(7), 3))) == list(range(7))


Batches and flattening
[(0, 1, 2), (3, 4, 5), (6,)]
[(0, 1, 2), (3, 4, 5), (6,)]
[0, 1, 2, 3, 4, 5, 6]


## Problem 9 — Prime Numbers as a Reusable Iterable

**Task**

Write `Primes(limit)` that yields all prime numbers less than `limit`. The iterable must be reusable and lazy.

Add tests for edge cases:

- `limit <= 2`
- repeated iteration
- concurrent iteration

**Best practice**: keep primality helper functions pure and keep iteration state local to `__iter__`.

In [10]:
def is_prime(n: int) -> bool:
    if n < 2:
        return False
    if n == 2:
        return True
    if n % 2 == 0:
        return False
    factor = 3
    while factor * factor <= n:
        if n % factor == 0:
            return False
        factor += 2
    return True


class Primes:
    def __init__(self, limit: int):
        if limit < 0:
            raise ValueError("limit must be >= 0")
        self.limit = limit

    def __iter__(self):
        for candidate in range(2, self.limit):
            if is_prime(candidate):
                yield candidate


section("Reusable primes iterable")
primes_under_30 = Primes(30)
print(list(primes_under_30))
print(list(primes_under_30))

p1 = iter(primes_under_30)
p2 = iter(primes_under_30)
print(next(p1), next(p1), next(p2))

assert list(Primes(2)) == []
assert list(Primes(10)) == [2, 3, 5, 7]
assert list(primes_under_30) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]


Reusable primes iterable
[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
2 3 2


## Problem 10 — Infinite Generator Recipe Wrapped as a Finite View

**Task**

Create an infinite generator function `arithmetic_progression(start, step)`. Then create a reusable iterable `FirstNProgression(start, step, n)` that yields the first `n` values.

**Best practice**: infinite generators are fine, but consumers should usually limit them with `islice`, a sentinel condition, or a finite wrapper.

In [11]:
def arithmetic_progression(start=0, step=1):
    current = start
    while True:
        yield current
        current += step


class FirstNProgression:
    def __init__(self, start: int, step: int, n: int):
        if n < 0:
            raise ValueError("n must be >= 0")
        self.start = start
        self.step = step
        self.n = n

    def __iter__(self):
        return islice(arithmetic_progression(self.start, self.step), self.n)


section("Finite reusable view over infinite recipe")
seq = FirstNProgression(start=10, step=5, n=6)
print(list(seq))
print(list(seq))

assert list(seq) == [10, 15, 20, 25, 30, 35]
assert list(FirstNProgression(0, -2, 5)) == [0, -2, -4, -6, -8]


Finite reusable view over infinite recipe
[10, 15, 20, 25, 30, 35]
[10, 15, 20, 25, 30, 35]


## Problem 11 — Lazy Memoized Iterable

**Task**

Create `MemoizedIterable(factory)` that:

1. Accepts a zero-argument generator factory.
2. Caches values as they are produced.
3. Lets later iterations replay cached values without recomputing them.
4. Continues the underlying source only once when needed.

This is more advanced than simply returning a fresh generator every time. It is useful when values are expensive to compute.

**Best practice**: be careful with shared mutable cache and a single underlying source; document that this is not thread-safe.

In [12]:
class MemoizedIterable:
    def __init__(self, factory: Callable[[], Iterator]):
        self.factory = factory
        self.cache = []
        self.source = None
        self.exhausted = False

    def __iter__(self):
        index = 0

        # Replay already-computed values first.
        while index < len(self.cache):
            yield self.cache[index]
            index += 1

        # Then extend the shared cache from the single source.
        if self.exhausted:
            return

        if self.source is None:
            self.source = iter(self.factory())

        for value in self.source:
            self.cache.append(value)
            index += 1
            yield value

        self.exhausted = True


def noisy_squares(n):
    for i in range(n):
        print(f"computing {i} ** 2")
        yield i ** 2


section("Memoized iterable")
memo = MemoizedIterable(lambda: noisy_squares(5))

print("first partial pass")
it = iter(memo)
print(next(it))
print(next(it))

print("second full pass replays first two, computes rest")
print(list(memo))

print("third full pass is fully cached")
print(list(memo))

assert list(memo) == [0, 1, 4, 9, 16]
assert list(memo) == [0, 1, 4, 9, 16]


Memoized iterable
first partial pass
computing 0 ** 2
0
computing 1 ** 2
1
second full pass replays first two, computes rest
computing 2 ** 2
computing 3 ** 2
computing 4 ** 2
[0, 1, 4, 9, 16]
third full pass is fully cached
[0, 1, 4, 9, 16]


## Problem 12 — `itertools.tee`: Useful, but Not a Reusable Iterable

**Task**

Use `tee` to split a single generator into two iterators. Show that:

1. The two tee objects can be consumed independently.
2. They are still one-shot iterators.
3. A reusable class is often clearer when you need repeated full passes.

**Best practice**: `tee` is useful for splitting one pass, not for designing a reusable collection-like object.

In [13]:
section("Using tee")
one_shot = squares_gen(5)
a, b = tee(one_shot, 2)

print(next(a))
print(next(a))
print(list(b))  # b still sees all values because tee buffered what a consumed
print(list(a))
print(list(a))  # a is now exhausted
print(list(b))  # b is now exhausted

section("Reusable class remains simpler for repeated passes")
sq = Squares(5)
print(list(sq))
print(list(sq))

assert list(Squares(5)) == [0, 1, 4, 9, 16]


Using tee
0
1
[0, 1, 4, 9, 16]
[4, 9, 16]
[]
[]

Reusable class remains simpler for repeated passes
[0, 1, 4, 9, 16]
[0, 1, 4, 9, 16]


## Problem 13 — Design a Reusable Windowed Iterable

**Task**

Create `SlidingWindows(iterable, width)` that yields overlapping tuples.

Example:

```python
list(SlidingWindows([10, 20, 30, 40], 3))
# [(10, 20, 30), (20, 30, 40)]
```

Make it reusable when the source is reusable.

**Best practice**: validate parameters early; create the source iterator inside `__iter__`.

In [14]:
from collections import deque


class SlidingWindows:
    def __init__(self, iterable, width: int):
        if width <= 0:
            raise ValueError("width must be positive")
        self.iterable = iterable
        self.width = width

    def __iter__(self):
        iterator = iter(self.iterable)
        window = deque(islice(iterator, self.width), maxlen=self.width)
        if len(window) < self.width:
            return
        yield tuple(window)

        for item in iterator:
            window.append(item)
            yield tuple(window)


section("Sliding windows")
windows = SlidingWindows([10, 20, 30, 40, 50], 3)
print(list(windows))
print(list(windows))

assert list(SlidingWindows([10, 20, 30, 40], 3)) == [(10, 20, 30), (20, 30, 40)]
assert list(SlidingWindows([1, 2], 3)) == []
assert list(windows) == [(10, 20, 30), (20, 30, 40), (30, 40, 50)]


Sliding windows
[(10, 20, 30), (20, 30, 40), (30, 40, 50)]
[(10, 20, 30), (20, 30, 40), (30, 40, 50)]


## Problem 14 — Build a Small Iterable Test Suite

**Task**

Write a helper `assert_reusable_iterable(obj, expected)` that checks:

1. `obj` is iterable.
2. `obj` is not its own iterator.
3. Two full passes produce the same result.
4. Two concurrent iterators do not unexpectedly share state.

Then run it against several classes from this notebook.

**Best practice**: test behavior, not just implementation details.

In [15]:
def assert_reusable_iterable(obj, expected):
    assert isinstance(obj, Iterable), f"{obj!r} is not iterable"
    assert iter(obj) is not obj, f"{obj!r} appears to be its own iterator"

    first = list(obj)
    second = list(obj)
    assert first == expected, f"first pass mismatch: {first!r}"
    assert second == expected, f"second pass mismatch: {second!r}"

    left = iter(obj)
    right = iter(obj)

    if expected:
        left_first = next(left)
        right_first = next(right)
        assert left_first == expected[0]
        assert right_first == expected[0], "concurrent iterator did not start fresh"

    return True


section("Reusable iterable test suite")
checks = [
    (Squares(5), [0, 1, 4, 9, 16]),
    (Countdown(3), [3, 2, 1, 0]),
    (CleanIntegers(["1", "#x", "2"]), [1, 2]),
    (Primes(12), [2, 3, 5, 7, 11]),
    (FirstNProgression(1, 3, 4), [1, 4, 7, 10]),
]

for obj, expected in checks:
    print(type(obj).__name__, assert_reusable_iterable(obj, expected))


Reusable iterable test suite
Squares True
Countdown True
CleanIntegers True
Primes True
FirstNProgression True


## Problem 15 — Capstone: Reusable Event Stream Query Object

**Task**

Build a reusable iterable `EventQuery` over event dictionaries. It should support:

1. Filtering by event type.
2. Filtering by minimum severity.
3. Mapping each event to a compact tuple.
4. Repeated full iteration.
5. Independent concurrent iterators.

Use generator helper functions internally.

**Best practice**: compose small generator stages, and make the public object reusable by returning a fresh pipeline in `__iter__`.

In [16]:
events = [
    {"id": 1, "type": "login", "user": "ada", "severity": 1},
    {"id": 2, "type": "error", "user": "linus", "severity": 5},
    {"id": 3, "type": "error", "user": "grace", "severity": 3},
    {"id": 4, "type": "logout", "user": "ada", "severity": 1},
    {"id": 5, "type": "error", "user": "ada", "severity": 4},
]


def filter_event_type(rows, event_type: Optional[str]):
    for row in rows:
        if event_type is None or row["type"] == event_type:
            yield row


def filter_min_severity(rows, minimum: int):
    for row in rows:
        if row["severity"] >= minimum:
            yield row


def compact_event(rows):
    for row in rows:
        yield (row["id"], row["user"], row["severity"])


class EventQuery:
    def __init__(self, rows, event_type: Optional[str] = None, min_severity: int = 0):
        self.rows = tuple(rows)  # stable snapshot
        self.event_type = event_type
        self.min_severity = min_severity

    def __iter__(self):
        rows = iter(self.rows)
        rows = filter_event_type(rows, self.event_type)
        rows = filter_min_severity(rows, self.min_severity)
        rows = compact_event(rows)
        return rows

    def __repr__(self):
        return (
            f"EventQuery(event_type={self.event_type!r}, "
            f"min_severity={self.min_severity!r}, rows={len(self.rows)})"
        )


section("Capstone event query")
query = EventQuery(events, event_type="error", min_severity=4)
print(query)
print(list(query))
print(list(query))

left = iter(query)
right = iter(query)
print("left:", next(left))
print("right:", next(right))
print("left rest:", list(left))
print("right rest:", list(right))

expected = [(2, "linus", 5), (5, "ada", 4)]
assert list(query) == expected
assert_reusable_iterable(query, expected)


Capstone event query
EventQuery(event_type='error', min_severity=4, rows=5)
[(2, 'linus', 5), (5, 'ada', 4)]
[(2, 'linus', 5), (5, 'ada', 4)]
left: (2, 'linus', 5)
right: (2, 'linus', 5)
left rest: [(5, 'ada', 4)]
right rest: [(5, 'ada', 4)]


True

## Summary Checklist

When turning a generator into a reusable iterable, check these points:

- Do not store a generator object if you need repeated passes.
- Store the generator function and arguments, or store source data and rebuild the pipeline in `__iter__`.
- `__iter__` should usually return a new iterator each time.
- Keep iteration state local to the generator returned by `__iter__`.
- Decide whether the iterable should be **live** or **snapshot-based**.
- Test repeated iteration and concurrent iteration.
- Use `itertools.tee` for splitting one pass, not as a general reusable iterable design.
